In [6]:
#   df_ml  → dummies version for OLS, Ridge, LASSO, Elastic Net
#   df_cat → raw categoricals for XGBoost feature importance
#   df_ols → df_ml with additional OLS-specific features (remaining_lease_sq for nonlinearity)

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df      = pd.read_parquet("../cleaned_datasets/resale_with_all_features_dummies.parquet")
df_raw  = pd.read_parquet("../cleaned_datasets/resale_with_all_features.parquet")
schools = pd.read_csv("../cleaned_datasets/schools_tiered.csv")

In [7]:
# 1. Parse remaining_lease to numeric 
def parse_remaining_lease(val):
    val = str(val).strip()
    if "year" in val:
        parts  = val.split()
        years  = int(parts[0])
        months = int(parts[2]) if len(parts) >= 4 else 0
        return years + months / 12
    return float(val)

df["remaining_lease_numeric"] = df["remaining_lease"].apply(parse_remaining_lease)

# 2. Log-transform target 
assert (df["resale_price"] > 0).all(), "Non-positive prices found"
df["log_resale_price"] = np.log(df["resale_price"])

# 3. Cap walking_dist_mall_m outlier 
# max of 20,885m is a geocoding error
cap_mall = df["walking_dist_mall_m"].quantile(0.99)
n_capped = (df["walking_dist_mall_m"] > cap_mall).sum()
df["walking_dist_mall_m"] = df["walking_dist_mall_m"].clip(upper=cap_mall)

# 4. Treatment variable
# binary: 1 if within 1km of Tier-1 school (MOE Phase 2B cutoff)
df["near_tier1_1km"] = (df["num_tier1_schools_1km"] >= 1).astype(int)

# 5. Additional OLS features
# remaining_lease has nonlinear effect (convex below ~60 years), squared term captures this for linear models
df["remaining_lease_sq"] = df["remaining_lease_numeric"] ** 2

# 6. Drop non-features
non_features_drop = [
    "month",            
    "year",             
    "geometry",         
    "postal_code",      
    "remaining_lease",  
    "resale_price",     
]

# 7. df_ml (XGBoost feature importance)
df_ml = df.drop(columns=non_features_drop, errors="ignore").copy()
df_ml.to_csv("../cleaned_datasets/df_ml.csv", index=False)


# 8. df_ols (OLS / Ridge / LASSO / Elastic Net)
df_ols = df.drop(
    columns=non_features_drop,
    errors="ignore"
).copy()

df_ols.to_csv("../cleaned_datasets/df_ols.csv", index=False)

# 9. Build df_cat (XGBoost with raw categoricals)
df_raw["remaining_lease_numeric"] = df_raw["remaining_lease"].apply(
    parse_remaining_lease
)

assert (df_raw["resale_price"] > 0).all(), "Non-positive prices found"
df_raw["log_resale_price"] = np.log(df_raw["resale_price"])

cap_mall_raw              = df_raw["walking_dist_mall_m"].quantile(0.99)
df_raw["walking_dist_mall_m"] = df_raw["walking_dist_mall_m"].clip(upper=cap_mall_raw)

# drop non-features for df_cat
non_features_drop_cat = non_features_drop + [
    "num_tier1_schools_1km", "num_tier1_schools_2km",
    "num_tier2_schools_1km", "num_tier2_schools_2km",
    "num_schools_1km",       "num_schools_2km",
]

df_cat = df_raw.drop(columns=non_features_drop_cat, errors="ignore").copy()

# cast categoricals — XGBoost handles natively via enable_categorical
for col in ["town", "flat_type", "storey_range", "flat_model"]:
    if col in df_cat.columns:
        df_cat[col] = df_cat[col].astype("category")

df_cat.to_pickle("../cleaned_datasets/df_cat.pkl")